# Max tree height (Filtered) maping by random forest models in GEE

In [1]:
# import the libraries
import ee
import pandas as pd
import os
import numpy as np
from termcolor import colored # this is allocate colour and fonts type for the print title and text

In [2]:
#set the working directory of local drive for Grid search result table loading
# os.getcwd()
# os.chdir('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject')

In [3]:
# initialize the earth engine API
ee.Initialize(project='ee-sun520ying')

In [4]:
# Load the composite for root shoot ratio analysis
compositeRaw = ee.Image("users/leonidmoore/ForestBiomass/20200915_Forest_Biomass_Predictors_Image")
# get the projection
stdProj = compositeRaw.projection()

In [5]:
forestAgeData = ee.Image("users/leonidmoore/ForestBiomass/20251218_ForestAge_SoilMoisture_Image")
print(colored('Band in Forest age data:\n', 'blue', attrs=['bold']),forestAgeData.bandNames().getInfo())
# we choose the band "forest_age_TC030" for following modeling
# forestAge = forestAgeData.select(['ForestAge']).reproject(crs=stdProj).rename('ForestAge')
# soilmoisture = forestAgeData.select(['SoilMoisture']).reproject(crs=stdProj).rename('SoilMoisture')

Band in Forest age data:
 ['ForestAge', 'SoilMoisture']


In [6]:
# load the additional layers and uniform the projection
# soilmoisture = ee.Image('users/haozhima95/wld_soil_moisture').reproject(crs=stdProj).rename('SoilMoisture')
waterAvail = compositeRaw.select("CHELSA_Annual_Precipitation").subtract(compositeRaw.select("PET")).rename("WaterAvailability")
ET_Ver3 = ee.Image('users/leonidmoore/Evapotranspiration_0_Ver_3_2023').rename('ET0').reproject(stdProj)
CHELSA_vpd = ee.Image("projects/ee-sun520ying/assets/CHELSA_vpd_mean_1981-2010_V_2_1").rename('CHELSA_vpd')
compositeImg = compositeRaw.addBands(forestAgeData).addBands(ET_Ver3).addBands(waterAvail).addBands(CHELSA_vpd)
# check the band names in the composite
compositeBandNames = compositeImg.bandNames()
print(colored('Band in composite:\n', 'blue', attrs=['bold']),compositeBandNames.getInfo())

Band in composite:
 ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvCloudCover_MODCF_interannualSD', 'EarthEnvCloudCover_MODCF_intraannualSD', 'EarthEnvCloudCover_MODCF_meanannual', 'EarthEnvTopoMed_AspectCosine', 'EarthEnvTopoMed_AspectSine', 'EarthEnv

In [7]:
# define the boundary geography reference
unboundedGeo = ee.Geometry.Polygon([-180, 88, 0, 88, 180, 88, 180, -88, 0, -88, -180, -88], None, False)

### Check the data structure of the paramter tables

In [8]:
# load the grid searh table from R
parameterTable = pd.read_csv('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject/ModelOptimization/GridSearchResultsGEE/Tree_Height_filtered_Buffer_Based_20260407_Subsample_Grid_Search_Seed_0.csv', float_precision='round_trip')
# show the structure by head function
print(colored('The head of the table: \n', 'blue', attrs=['bold']))
parameterTable.head()

The head of the table: 



,Unnamed: 0,Mean_R2,ModelName,StDev_R2,type,numberOfTrees,variablesPerSplit,minLeafPopulation,bagFraction,maxNodes
0,0,0.436523,GridSeach_Model_15_2_90,0.074232,Classifier.smileRandomForest,500,15,2,0.632,90


In [9]:
# define the list of predictors
predictorsSelected = ["Aridity_Index",
                      "CHELSA_Annual_Mean_Temperature",
                      "CHELSA_Annual_Precipitation",
                      "CHELSA_Isothermality",
                      "CHELSA_Max_Temperature_of_Warmest_Month",
                      "CHELSA_Mean_Diurnal_Range",
                      "CHELSA_Mean_Temperature_of_Coldest_Quarter",
                      "CHELSA_Mean_Temperature_of_Driest_Quarter",
                      "CHELSA_Mean_Temperature_of_Warmest_Quarter",
                      "CHELSA_Mean_Temperature_of_Wettest_Quarter",
                      "CHELSA_Min_Temperature_of_Coldest_Month",
                      "CHELSA_Precipitation_Seasonality",
                      "CHELSA_Precipitation_of_Coldest_Quarter",
                      "CHELSA_Precipitation_of_Driest_Month",
                      "CHELSA_Precipitation_of_Driest_Quarter",
                      "CHELSA_Precipitation_of_Warmest_Quarter",
                      "CHELSA_Precipitation_of_Wettest_Month",
                      "CHELSA_Precipitation_of_Wettest_Quarter",
                      "CHELSA_Temperature_Annual_Range",
                      "CHELSA_Temperature_Seasonality",
                      "Depth_to_Water_Table",
                      "EarthEnvCloudCover_MODCF_interannualSD",
                      "EarthEnvCloudCover_MODCF_intraannualSD",
                      "EarthEnvCloudCover_MODCF_meanannual",
                      "EarthEnvTopoMed_Eastness",
                      "EarthEnvTopoMed_Elevation",
                      "EarthEnvTopoMed_Northness",
                      "EarthEnvTopoMed_ProfileCurvature",
                      "EarthEnvTopoMed_Roughness",
                      "EarthEnvTopoMed_Slope",
                      "EarthEnvTopoMed_TopoPositionIndex",
                      "SG_Absolute_depth_to_bedrock",
                      "WorldClim2_SolarRadiation_AnnualMean",
                      "WorldClim2_WindSpeed_AnnualMean",
                      "WorldClim2_H2OVaporPressure_AnnualMean",
                      "NDVI",
                      "EVI",
                      "Lai",
                      "Fpar",
                      "Npp",
                      "Tree_Density",
                      "PET",
                      "SG_Clay_Content_0_100cm",
                      "SG_Coarse_fragments_0_100cm",
                      "SG_Sand_Content_0_100cm",
                      "SG_Silt_Content_0_100cm",
                      "SG_Soil_pH_H2O_0_100cm",
                      "LandCoverClass_Cultivated_and_Managed_Vegetation",
                      "LandCoverClass_Urban_Builtup",
                      "Human_Disturbance",
                      "PresentTreeCover",
                      "Nitrogen",
                      "cropland",
                      "grazing",
                      "pasture",
                      "rangeland",
                      "cnRatio",
                      "Cation",
                      "SoilMoisture",
                      "ForestAge",
                      "Organic_Carbon",
                      "GlobBiomass_GrowingStockVolume",
                      "Human_Development_Percentage",
                      "HumanFootprint",
                      "Rainfall_Erosivity",
                      "WDPA",
                      "Fire_Frequency",
                      "WaterAvailability",
                      'CHELSA_vpd']
# show the predictors
print(colored('The predictors are:', 'blue', attrs=['bold']),predictorsSelected)

The predictors are: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvCloudCover_MODCF_interannualSD', 'EarthEnvCloudCover_MODCF_intraannualSD', 'EarthEnvCloudCover_MODCF_meanannual', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoM

### After you run the chunk below, please check the status of the maping tasks on your Google earth engine

In [11]:
# creat a GEE folder to save following sampled data
# ee.data.createAsset({'type': 'FOLDER'},'users/sun520ying/TreeStructureProject/EnsembleMaps')

In [10]:
# define a loop through the seed list
seedList = np.arange(0, 50, 1).tolist()
print(colored('The models are:', 'blue', attrs=['bold']),seedList)
print(colored('Model is running:\nWith paramter sets:', 'blue', attrs=['bold']))
# for seed in seedList: range(0,len(seedList))
for seed in seedList:
    # load the points data in shape file format
    maxTreeHeightPoints = ee.FeatureCollection('users/sun520ying/TreeStructureProject/TrainTables/TreeHeight_filtered_BufferZone_20260407_Subsampled_Train_table_Seed_'+str(seed))
    # check the information of the FeatureCollection
    # print(woodDensityTable.first().getInfo())
    # extract the environment covariates
    extractedVariableTable = compositeImg.select(predictorsSelected).reduceRegions(collection = maxTreeHeightPoints,
                                                                                     reducer = ee.Reducer.first())
    # show the str of the extracted data
    # print(extractedVariableTable.first().getInfo())
    # exclude the rows with null valus
    trainTable = extractedVariableTable.filter(ee.Filter.notNull(predictorsSelected))
    # print(trainTable.size().getInfo())
    parameterTable = pd.read_csv('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject/ModelOptimization/GridSearchResultsGEE/Tree_Height_filtered_Buffer_Based_20260407_Subsample_Grid_Search_Seed_'+str(seed)+'.csv', float_precision='round_trip')
    # not recomend to run the code below
    # print(parameterTable.head())
    # extract the paramters
    variablesPerSplitVal = int(parameterTable['variablesPerSplit'].iat[0]) # mtry
    minLeafPopulationVal = int(parameterTable['minLeafPopulation'].iat[0]) # minrow
    maxNodesVal = int(parameterTable['maxNodes'].iat[0]) # mac depth
    print('seed',seed,variablesPerSplitVal,minLeafPopulationVal,maxNodesVal)
    # define the random forest classifier
    rfClassifier = ee.Classifier.smileRandomForest(numberOfTrees = 500,
                                                   variablesPerSplit = variablesPerSplitVal, # mtry
                                                   minLeafPopulation = minLeafPopulationVal, # minrow
                                                   maxNodes = maxNodesVal, # max depth
                                                   bagFraction = 0.632,
                                                   seed = seed).setOutputMode('REGRESSION')
    trainedClassifier = rfClassifier.train(features = trainTable,
                                           classProperty = 'MaxHeight',
                                           inputProperties = predictorsSelected)
    # execute the prediction to generate the map
    predictedWoodDensityMap = compositeImg.classify(trainedClassifier)
    # print(predictedWoodDensityMap.getInfo())
    predictionExport = ee.batch.Export.image.toAsset(image = predictedWoodDensityMap,
                                                     description = '20260407_TreeHeight_Map_To_Asset_'+str(seed),
                                                     assetId = 'users/sun520ying/TreeStructureProject/PredictedMaps/Predicted_Max_TreeHeight_Filtered_BufferBased_20260407_Mean_Map_with_Seed_'+str(seed),
                                                     region = unboundedGeo,
                                                     crs = 'EPSG:4326',
                                                     crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                                     maxPixels = 1e13)

    # print(predictionExportAsset)
    # start the export task
    predictionExport.start()
    # show the task status
    predictionExport.status()

The models are: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Model is running:
With paramter sets:
seed 0 15 2 90
seed 1 18 2 80
seed 2 9 4 80
seed 3 21 2 100
seed 4 12 4 100
seed 5 9 2 100
seed 6 15 4 100
seed 7 18 4 100
seed 8 18 2 100
seed 9 9 2 100
seed 10 21 4 80
seed 11 21 2 100
seed 12 21 4 60
seed 13 21 2 90
seed 14 21 2 100
seed 15 6 4 100
seed 16 12 2 100
seed 17 15 4 100
seed 18 18 4 100
seed 19 21 2 100
seed 20 15 4 70
seed 21 18 2 100
seed 22 12 4 100
seed 23 21 2 70
seed 24 12 4 90
seed 25 9 4 100
seed 26 9 2 80
seed 27 12 2 100
seed 28 18 4 90
seed 29 18 2 100
seed 30 12 4 100
seed 31 9 2 100
seed 32 18 2 90
seed 33 18 2 90
seed 34 12 4 80
seed 35 18 2 90
seed 36 18 2 100
seed 37 18 4 80
seed 38 18 4 70
seed 39 12 2 70
seed 40 21 4 100
seed 41 12 6 70
seed 42 21 4 90
seed 43 18 2 80
seed 44 15 4 90
seed 45 21 2 80
seed 46 15 2 